# Benchmark Steering Vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/06_benchmark_vectors.ipynb)

This notebook systematically evaluates pre-trained BiPO steering vectors using the IPIP-NEO-120 personality inventory. We measure how each vector shifts the Big Five personality profile at various strengths.

**In this notebook you will:**
1. Download all 5 pre-trained English BiPO vectors
2. Run IPIP-NEO-120 benchmark at multiple strengths
3. Analyze cross-impact (how each vector affects all Big Five domains)
4. (Optional) Test API models with OpenRouter

**Requirements:** Colab free tier (T4 GPU). [HuggingFace token](https://huggingface.co/settings/tokens) with access to Llama-3.1-8B-Instruct.

**Time:** ~15 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if isinstance(hf_token, str) and hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your HuggingFace token: ")

# Optional: OpenRouter API key (for the API model section at the end)
try:
    from google.colab import userdata
    or_key = userdata.get("OPENROUTER_API_KEY")
    if isinstance(or_key, str) and or_key:
        os.environ["OPENROUTER_API_KEY"] = or_key
        print("Loaded OPENROUTER_API_KEY from Colab Secrets")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## Download Pre-trained Vectors

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

VECTOR_REPO = "dalekwon/bipo-steering-vectors"
VECTOR_DIR = Path("./vectors")
VECTOR_DIR.mkdir(exist_ok=True)

# Big Five-mapped vectors for inventory benchmarking
VECTORS = {
    "agreeableness": {
        "filename": "bipo_steering_english_agreeableness.safetensors",
        "big_five_domain": "A",
    },
    "neuroticism": {
        "filename": "bipo_steering_english_neuroticism.safetensors",
        "big_five_domain": "N",
    },
}

# Additional personality vectors
EXTRA_VECTORS = {
    "awfully_sweet": "bipo_steering_english_awfully_sweet.safetensors",
    "paranoid": "bipo_steering_english_paranoid.safetensors",
    "very_lascivious": "bipo_steering_english_very_lascivious.safetensors",
}

vector_paths = {}
for name, info in VECTORS.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=info["filename"],
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name} (Big Five: {info['big_five_domain']}): ready")

for name, filename in EXTRA_VECTORS.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=filename,
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name}: ready")

print(f"\nAll {len(vector_paths)} vectors downloaded.")

## IPIP-NEO-120 Benchmark

We use PSYCTL's `InventoryTester` to run the full IPIP-NEO-120 inventory at multiple steering strengths. This measures how each Big Five domain shifts when a steering vector is applied.

In [ ]:
from psyctl.core.benchmark.inventory_tester import InventoryTester

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
STRENGTHS = [1.0, 2.0, 3.0]
INVENTORY = "ipip_neo_120"

tester = InventoryTester()
all_results = {}

# Benchmark Big Five-mapped vectors
for trait_name, info in VECTORS.items():
    vpath = vector_paths[trait_name]
    print(f"\n{'='*60}")
    print(f"Benchmarking: {trait_name} (target domain: {info['big_five_domain']})")
    print(f"{'='*60}")

    trait_results = []
    for strength in STRENGTHS:
        print(f"  Testing strength={strength}...")
        result = tester.test_inventory(
            model=MODEL,
            steering_vector_path=vpath,
            inventory_name=INVENTORY,
            steering_strength=strength,
            target_trait=trait_name,
        )
        trait_results.append({"strength": strength, "result": result})

        # Print comparison
        comparison = result.get("comparison", {})
        if comparison:
            for code, d in comparison.items():
                marker = " <--" if code == info["big_five_domain"] else ""
                print(f"    {d['domain_name']:>20}: {d['change']:+.3f} ({d['percent_change']:+.1f}%){marker}")

    all_results[trait_name] = trait_results

print("\nBenchmark complete!")

## Cross-Impact Analysis

A good steering vector should primarily affect its target Big Five domain while minimizing unintended changes to other domains. Let's visualize this.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract cross-impact data at strength=2.0
domains = ["N", "E", "O", "A", "C"]
domain_names = {"N": "Neuroticism", "E": "Extraversion", "O": "Openness",
                "A": "Agreeableness", "C": "Conscientiousness"}

fig, axes = plt.subplots(1, len(VECTORS), figsize=(6 * len(VECTORS), 5), sharey=True)
if len(VECTORS) == 1:
    axes = [axes]

for idx, (trait_name, info) in enumerate(VECTORS.items()):
    ax = axes[idx]

    # Find strength=2.0 result
    for r in all_results[trait_name]:
        if r["strength"] == 2.0:
            comparison = r["result"].get("comparison", {})
            break
    else:
        comparison = {}

    changes = []
    colors = []
    for d in domains:
        if d in comparison:
            changes.append(comparison[d]["percent_change"])
            colors.append("#E85D75" if d == info["big_five_domain"] else "#4A90D9")
        else:
            changes.append(0)
            colors.append("#cccccc")

    ax.bar(range(len(domains)), changes, color=colors)
    ax.set_xticks(range(len(domains)))
    ax.set_xticklabels([domain_names[d] for d in domains], rotation=30, ha="right")
    ax.set_title(f"{trait_name} (target: {info['big_five_domain']})", fontweight="bold")
    ax.set_ylabel("% Change from Baseline" if idx == 0 else "")
    ax.axhline(y=0, color="grey", linewidth=0.5)

plt.suptitle("Cross-Impact: Strength = 2.0", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## (Optional) OpenRouter Inventory Test

Test multiple API models using OpenRouter. This sends personality inventory questions via chat API and parses Likert scale responses.

In [ ]:
# Uncomment to run OpenRouter inventory test on API models
#
# import re, time
# from psyctl.data.inventories import create_inventory
# from psyctl.models.openrouter_client import OpenRouterClient
#
# OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
# if not OPENROUTER_API_KEY:
#     print("Set OPENROUTER_API_KEY in Colab Secrets to use this section.")
# else:
#     client = OpenRouterClient(api_key=OPENROUTER_API_KEY)
#     inventory = create_inventory("rei_40")
#     questions = inventory.get_questions()
#
#     SYSTEM_PROMPT = (
#         "You are taking a personality assessment. For each statement, respond with ONLY "
#         "a single number from 1 to 5.\n\nScale:\n1 = Definitely not true\n"
#         "2 = Somewhat not true\n3 = Neither true nor untrue\n4 = Somewhat true\n"
#         "5 = Definitely true\n\nRespond with ONLY the number."
#     )
#
#     MODELS = [
#         ("anthropic/claude-sonnet-4", "Claude Sonnet 4"),
#         ("openai/gpt-4o", "GPT-4o"),
#     ]
#
#     for model_id, model_name in MODELS:
#         print(f"\nTesting {model_name}...")
#         domain_responses = {}
#         for q in questions:
#             prompt = f'Statement: "{q["text"]}"\nYour rating (1-5):'
#             _, resp = client.generate(
#                 prompt=prompt, model=model_id,
#                 temperature=0, max_tokens=16, system_prompt=SYSTEM_PROMPT,
#             )
#             match = re.search(r"\b([1-5])\b", resp.strip())
#             score = float(match.group(1)) if match else 3.0
#             if q["keyed"] == "minus":
#                 score = 6.0 - score
#             domain = q["domain"]
#             domain_responses.setdefault(domain, []).append(score)
#             time.sleep(0.3)
#
#         scores = inventory.calculate_scores(domain_responses)
#         print(f"  {model_name} REI-40 scores:")
#         for code in sorted(scores.keys()):
#             s = scores[code]
#             print(f"    {s['domain_name']:<25} raw={s['raw_score']:.1f}  z={s['z_score']:+.2f}  pctl={s['percentile']:.1f}%")

## Key Findings

- **Targeted impact**: BiPO vectors primarily shift their target Big Five domain, confirming they capture the intended personality direction
- **Cross-domain effects**: Some cross-talk is expected — e.g., agreeableness vectors may slightly increase extraversion
- **Strength scaling**: Higher strengths produce larger shifts, but very high values (>3.0) can degrade coherence
- **Recommended range**: Strengths between 1.0 and 2.5 typically give the best balance of personality shift and output quality

**Next steps:**
- [01_quickstart.ipynb](./01_quickstart.ipynb) — Try steering interactively
- [04_extract_vector.ipynb](./04_extract_vector.ipynb) — Extract your own vectors
- [GitHub](https://github.com/modulabs-personalab/psyctl) — Contribute new inventories or vectors